# FotoPapeleta Inteligente — Algoritmos de búsqueda BFS, DFS y A*

Universidad Tecnológica del Perú · APF2 · Metodología CRISP-DM

La red de cámaras se modela como un **grafo**: cada cámara es un nodo y cada tramo de vía es una arista con su
distancia (km) y su tiempo típico de viaje (min). Sobre ese grafo:

| Algoritmo | Para qué se usa en el proyecto |
|---|---|
| **BFS** | Cámaras alcanzables en *k* saltos: delimita dónde buscar nuevamente una placa leída |
| **DFS** (acotado) | Enumera las trayectorias posibles entre dos cámaras |
| **A\*** | Ruta de menor costo con heurística de línea recta (admisible); también la ruta más rápida según el tráfico |

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todas`. El notebook crea solo el paquete `grafo/` y las redes
en `data/`; no hay que subir archivos. Solo usa Python, `networkx` y `matplotlib`, que ya vienen en Colab.

Cada sección reproduce una prueba de la **Tabla II** del paper (P1–P7).

## 1. Crear el paquete `grafo` y las redes de cámaras
Es el mismo código del proyecto (`grafo/`), Python puro.

In [ ]:
import os
os.makedirs('grafo', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('Carpetas listas')

In [ ]:
%%writefile grafo/__init__.py
"""Paquete grafo: red de cámaras y búsqueda de evidencia (Módulos B y C).

Python puro, sin dependencias de Django ni de la BD: lo usan igual la web y el notebook.
"""
from .busqueda import a_estrella, bfs, costo_ruta, dfs
from .decision import evaluar_evento
from .evidencia import V_MAX_PLAUSIBLE_KMH, capturas_coherentes, clave, misma_placa, velocidad_implicita
from .heuristicas import nula, recta_km, recta_min
from .modelo import cargar, construir, validar
from .velocidad import velocidad_por_tramo

__all__ = ["a_estrella", "bfs", "costo_ruta", "dfs", "capturas_coherentes", "clave",
           "misma_placa", "nula", "recta_km", "recta_min", "cargar", "construir",
           "validar", "velocidad_por_tramo", "evaluar_evento", "velocidad_implicita",
           "V_MAX_PLAUSIBLE_KMH"]

In [ ]:
%%writefile grafo/modelo.py
"""Módulo B: carga, construcción y validación del grafo de cámaras."""
import json
import math
from pathlib import Path


def cargar(ruta):
    """Lee el JSON de la red y devuelve (grafo, nodos)."""
    datos = json.loads(Path(ruta).read_text(encoding="utf-8"))
    errores = validar(datos)
    if errores:
        raise ValueError("Grafo inválido:\n- " + "\n- ".join(errores))
    return construir(datos), datos["nodos"]


def construir(datos):
    """Lista de adyacencia dirigida: {camara: [(vecino, {km, min, limite_kmh}), ...]}.

    Una arista con "dirigida": false (vía de doble sentido) genera dos arcos.
    """
    grafo = {n: [] for n in datos["nodos"]}
    for a in datos["aristas"]:
        peso = {"km": a["km"], "min": a["min"], "limite_kmh": a.get("limite_kmh")}
        grafo[a["de"]].append((a["a"], peso))
        if not a.get("dirigida", False):
            grafo[a["a"]].append((a["de"], peso))
    return grafo


def validar(datos):
    """Devuelve la lista de errores; vacía si la red es consistente."""
    errores = []
    nodos = datos.get("nodos", {})
    for a in datos.get("aristas", []):
        tramo = f'{a.get("de")} -> {a.get("a")}'
        for extremo in (a.get("de"), a.get("a")):
            if extremo not in nodos:
                errores.append(f"{tramo}: la cámara {extremo} no existe")
        if a.get("km", 0) <= 0 or a.get("min", 0) <= 0:
            errores.append(f"{tramo}: km y min deben ser positivos")
        if errores:
            continue
        # La heurística de A* solo es admisible si la recta no supera al tramo.
        recta = distancia_recta_km(nodos[a["de"]], nodos[a["a"]])
        if recta is not None and recta > a["km"] + 1e-9:
            errores.append(f"{tramo}: la distancia en línea recta ({recta:.2f} km) supera al tramo ({a['km']} km)")
    return errores


def distancia_recta_km(p, q):
    """Distancia en línea recta entre dos cámaras.

    Usa coordenadas proyectadas (x_km, y_km) si existen; si no, Haversine con lat/lon.
    Devuelve None si la cámara no tiene coordenadas.
    """
    if "x_km" in p and "x_km" in q:
        return math.dist((p["x_km"], p["y_km"]), (q["x_km"], q["y_km"]))
    if "lat" in p and "lat" in q:
        lat1, lon1, lat2, lon2 = map(math.radians, (p["lat"], p["lon"], q["lat"], q["lon"]))
        h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
        return 2 * 6371 * math.asin(math.sqrt(h))
    return None

In [ ]:
%%writefile grafo/heuristicas.py
"""Heurísticas para A*. Deben estar en las mismas unidades que el peso y no sobreestimar."""
from .modelo import distancia_recta_km


def nula(a, b):
    """h = 0: A* se comporta como Dijkstra."""
    return 0


def recta_km(nodos):
    """Distancia en línea recta (km). Admisible para peso="km"."""
    def h(a, b):
        d = distancia_recta_km(nodos[a], nodos[b])
        return 0 if d is None else d
    return h


def recta_min(nodos, vmax_kmh=80):
    """Tiempo mínimo (min) recorriendo la recta a la velocidad máxima de la red. Admisible para peso="min"."""
    km = recta_km(nodos)
    return lambda a, b: km(a, b) / vmax_kmh * 60

In [ ]:
%%writefile grafo/busqueda.py
"""Módulo C: BFS, DFS acotado y A* sobre el grafo de cámaras."""
import heapq
from collections import deque, namedtuple

from .heuristicas import nula

ResultadoRuta = namedtuple("ResultadoRuta", "ruta costo explorados")


def bfs(grafo, origen, k):
    """Cámaras alcanzables en <= k saltos: {camara: saltos} (incluye el origen con 0)."""
    visitados, cola = {origen: 0}, deque([origen])
    while cola:
        actual = cola.popleft()
        if visitados[actual] == k:
            continue
        for vecino, _ in grafo[actual]:
            if vecino not in visitados:
                visitados[vecino] = visitados[actual] + 1
                cola.append(vecino)
    return visitados


def dfs(grafo, origen, destino, max_prof=6, max_rutas=50):
    """Rutas simples de origen a destino, acotadas por profundidad y cantidad."""
    rutas, camino = [], []

    def visitar(actual):
        camino.append(actual)
        if actual == destino:
            rutas.append(list(camino))
        elif len(camino) <= max_prof and len(rutas) < max_rutas:
            for vecino, _ in grafo[actual]:
                if vecino not in camino:
                    visitar(vecino)
        camino.pop()

    visitar(origen)
    return rutas


def a_estrella(grafo, origen, destino, h=nula, peso="km", traza=None):
    """Ruta de menor costo. Devuelve ResultadoRuta(ruta, costo, explorados); ruta=None si no hay camino.

    Si se pasa una lista en `traza`, se le agregan los nodos en el orden en que se exploran (laboratorio).
    """
    abierto = [(h(origen, destino), origen)]
    g, padre, cerrados, explorados = {origen: 0}, {}, set(), 0
    while abierto:
        _, actual = heapq.heappop(abierto)
        if actual in cerrados:  # entrada obsoleta de la cola
            continue
        cerrados.add(actual)
        explorados += 1
        if traza is not None:
            traza.append(actual)
        if actual == destino:
            ruta = [actual]
            while ruta[-1] in padre:
                ruta.append(padre[ruta[-1]])
            return ResultadoRuta(ruta[::-1], g[destino], explorados)
        for vecino, w in grafo[actual]:
            costo = g[actual] + w[peso]
            if costo < g.get(vecino, float("inf")):
                g[vecino], padre[vecino] = costo, actual
                heapq.heappush(abierto, (costo + h(vecino, destino), vecino))
    return ResultadoRuta(None, float("inf"), explorados)


def costo_ruta(grafo, ruta, peso="km"):
    """Suma del peso de los arcos de una ruta."""
    total = 0
    for a, b in zip(ruta, ruta[1:]):
        total += next(w[peso] for v, w in grafo[a] if v == b)
    return total

In [ ]:
%%writefile grafo/evidencia.py
"""Vinculación de capturas entre cámaras: coincidencia de placa y filtro temporal."""
from .busqueda import a_estrella, bfs

# Caracteres que el OCR suele confundir, llevados a una forma común.
_CONFUSIONES = str.maketrans({"O": "0", "Q": "0", "D": "0", "I": "1", "L": "1",
                              "B": "8", "S": "5", "Z": "2", "G": "6"})


def normalizar(placa):
    return placa.upper().replace("-", "").replace(" ", "")


def clave(placa):
    """Forma de la placa sin caracteres confundibles; se indexa en la BD como placa_clave."""
    return normalizar(placa).translate(_CONFUSIONES)


def _un_caracter_distinto(a, b):
    return len(a) == len(b) and sum(x != y for x, y in zip(a, b)) == 1


def misma_placa(a, b):
    """'confirmada', 'a_verificar' o None.

    'a_verificar' si difieren solo en caracteres confundibles (B/8, O/0...) o en UN solo
    carácter cualquiera (error típico del OCR entre cámaras con distinta luz). Esas
    coincidencias nunca se emiten sin revisión humana y pasan además por el filtro temporal.
    """
    na, nb = normalizar(a), normalizar(b)
    if na == nb:
        return "confirmada"
    if clave(a) == clave(b) or _un_caracter_distinto(na, nb):
        return "a_verificar"
    return None


V_MAX_PLAUSIBLE_KMH = 200  # ningún vehículo recorre la ciudad más rápido que esto


def velocidad_implicita(grafo, cam_a, ts_a, cam_b, ts_b):
    """(km mínimos entre las cámaras, km/h necesarios para cubrirlos en el tiempo observado)."""
    km = a_estrella(grafo, cam_a, cam_b, peso="km").costo
    horas = abs((ts_b - ts_a).total_seconds()) / 3600
    return km, (float("inf") if horas == 0 else km / horas)


def capturas_coherentes(grafo, evento, capturas, k=2, v_max_kmh=V_MAX_PLAUSIBLE_KMH):
    """Capturas de la misma placa en cámaras cercanas cuyo tiempo es físicamente posible.

    `evento` y cada captura son dicts con: placa, camara, ts (datetime).
    Una captura se acepta si, para recorrer la distancia mínima (A* en km) en el tiempo
    observado, bastaría una velocidad <= v_max_kmh. Así un auto muy rápido (p. ej. 100 km/h
    en zona de 60) se evalúa como posible infracción, y solo lo irreal (> 200 km/h) se trata
    como captura imposible (placa clonada o error de lectura).
    Devuelve [(captura, 'confirmada'|'a_verificar'), ...].
    """
    candidatas = bfs(grafo, evento["camara"], k)
    aceptadas = []
    for c in capturas:
        if c["camara"] == evento["camara"] or c["camara"] not in candidatas:
            continue
        coincidencia = misma_placa(evento["placa"], c["placa"])
        if coincidencia is None:
            continue
        _, v = velocidad_implicita(grafo, c["camara"], c["ts"], evento["camara"], evento["ts"])
        if v <= v_max_kmh:
            aceptadas.append((c, coincidencia))
    return aceptadas

In [ ]:
%%writefile grafo/velocidad.py
"""Velocidad promedio por tramo (section speed), calculada a favor del conductor."""
from dataclasses import dataclass

from .busqueda import a_estrella
from .heuristicas import nula


@dataclass
class ResultadoVelocidad:
    ruta: list
    km: float
    v_medida: float
    v_conservadora: float
    limite: float
    exceso: float
    infraccion: bool


def velocidad_por_tramo(grafo, cam_a, t_a, cam_b, t_b, h=nula, limite=None,
                        error_reloj_s=2, tolerancia_kmh=5):
    """Velocidad mínima posible entre dos capturas de la misma placa.

    Usa la distancia mínima (A*) y el tiempo máximo (Δt + 2ε). Si no se pasa
    `limite`, toma el límite más alto de la ruta (también a favor del conductor).
    Devuelve None si no hay ruta o si Δt <= 0.
    """
    res = a_estrella(grafo, cam_a, cam_b, h, peso="km")
    dt = (t_b - t_a).total_seconds()
    if res.ruta is None or dt <= 0:
        return None
    if limite is None:
        limite = max(w["limite_kmh"] for a, b in zip(res.ruta, res.ruta[1:])
                     for v, w in grafo[a] if v == b)
    v_medida = res.costo / (dt / 3600)
    v_cons = res.costo / ((dt + 2 * error_reloj_s) / 3600)
    return ResultadoVelocidad(res.ruta, res.costo, round(v_medida, 1), round(v_cons, 1),
                              limite, round(v_cons - limite, 1), v_cons > limite + tolerancia_kmh)

In [ ]:
%%writefile grafo/decision.py
"""Decisión ante un evento nuevo: ¿hay infracción de velocidad o una captura imposible?

Es lo que hace el sistema cada vez que una cámara reporta una placa. Función pura: la usan
la API de Django y la demo, sin depender de ninguna de las dos.
"""
from .evidencia import V_MAX_PLAUSIBLE_KMH, capturas_coherentes, misma_placa, velocidad_implicita
from .heuristicas import recta_km
from .velocidad import velocidad_por_tramo


def evaluar_evento(grafo, nodos, evento, capturas_previas, k=2, v_max_kmh=V_MAX_PLAUSIBLE_KMH,
                   **opciones_velocidad):
    """Compara un evento con las capturas previas de la misma placa.

    `evento` y cada captura: dict con placa, camara, ts (datetime) y un identificador `id`.
    Devuelve dict con tres listas:
      infracciones   : tramo desde la captura más cercana en el tiempo, por encima del límite + tolerancia
      sin_infraccion : tramos dentro del límite
      alertas        : misma placa en otra cámara en un tiempo físicamente imposible
                       (haría falta ir a más de v_max_kmh por la ruta más corta)
    """
    h = recta_km(nodos)
    aceptadas = capturas_coherentes(grafo, evento, capturas_previas, k=k, v_max_kmh=v_max_kmh)
    ids_ok = {c["id"] for c, _ in aceptadas}
    resultado = {"infracciones": [], "sin_infraccion": [], "alertas": []}

    for c in capturas_previas:
        if c["id"] in ids_ok or c["camara"] == evento["camara"]:
            continue
        coincidencia = misma_placa(c["placa"], evento["placa"])
        if coincidencia is None:
            continue
        km, v = velocidad_implicita(grafo, c["camara"], c["ts"], evento["camara"], evento["ts"])
        if km == float("inf") or v <= v_max_kmh:
            continue  # sin conexión en la red, o posible pero fuera del vecindario k: nada que alertar
        observado = abs((evento["ts"] - c["ts"]).total_seconds()) / 60
        resultado["alertas"].append({
            "tipo": "captura_imposible", "placa": evento["placa"], "coincidencia": coincidencia,
            "captura_anterior": c["id"], "captura_actual": evento["id"],
            "de": c["camara"], "a": evento["camara"], "km_minimos": km,
            "minutos_observados": round(observado, 2), "velocidad_necesaria_kmh": round(v, 1),
            "velocidad_maxima_plausible_kmh": v_max_kmh, "minimo_plausible_min": round(km / v_max_kmh * 60, 2),
            "motivo": "Captura físicamente imposible: posible placa clonada o error de lectura"})

    # Solo la captura más cercana en el tiempo: con A -> B -> C se evalúan A-B y luego B-C, nunca A-C,
    # que repetiría un recorrido ya evaluado (control por tramos consecutivos).
    aceptadas = sorted(aceptadas, key=lambda x: abs((evento["ts"] - x[0]["ts"]).total_seconds()))[:1]
    for c, coincidencia in aceptadas:
        anterior, actual = sorted((c, evento), key=lambda x: x["ts"])
        v = velocidad_por_tramo(grafo, anterior["camara"], anterior["ts"], actual["camara"], actual["ts"], h,
                                **opciones_velocidad)
        if v is None:
            continue
        fila = {"placa": evento["placa"], "coincidencia": coincidencia,
                "captura_anterior": anterior["id"], "captura_actual": actual["id"],
                "ruta": v.ruta, "km": v.km, "segundos": (actual["ts"] - anterior["ts"]).total_seconds(),
                "v_medida": v.v_medida, "v_conservadora": v.v_conservadora,
                "limite": v.limite, "exceso": v.exceso}
        resultado["infracciones" if v.infraccion else "sin_infraccion"].append(fila)
    return resultado

Redes de cámaras:
- `grafo_camaras.json`: red **esquemática** de 6 cámaras (Tabla I del paper), para la evaluación algorítmica.
- `grafo_camaras_real.json`: las mismas 6 cámaras con ubicaciones de OpenStreetMap y tramos por calles reales (OSRM).
- `red_transito.json`: las **15 cámaras públicas** de tránsito de Lima usadas para el conteo vehicular.

In [ ]:
%%writefile data/grafo_camaras.json
{
  "version": "2026-10-15",
  "nota": "Coordenadas x_km/y_km proyectadas y esquemáticas. Reemplazar por lat/lon reales (Google Maps / OSM).",
  "nodos": {
    "AV-01": {"nombre": "Av. Javier Prado", "x_km": 0.0, "y_km": 0.0},
    "AV-08": {"nombre": "Av. Arequipa", "x_km": 2.9, "y_km": 0.0},
    "AV-15": {"nombre": "Av. Angamos", "x_km": 7.2, "y_km": 0.0},
    "AV-02": {"nombre": "Av. Salaverry", "x_km": 0.0, "y_km": -2.0},
    "AV-12": {"nombre": "Av. Brasil", "x_km": 2.9, "y_km": -3.6},
    "PN-04": {"nombre": "Panamericana Norte", "x_km": 8.5, "y_km": -5.5}
  },
  "aristas": [
    {"de": "AV-01", "a": "AV-08", "km": 3.0, "min": 4, "limite_kmh": 60, "dirigida": false},
    {"de": "AV-01", "a": "AV-02", "km": 2.1, "min": 3, "limite_kmh": 60, "dirigida": false},
    {"de": "AV-08", "a": "AV-15", "km": 4.5, "min": 6, "limite_kmh": 60, "dirigida": false},
    {"de": "AV-08", "a": "AV-12", "km": 3.8, "min": 5, "limite_kmh": 60, "dirigida": false},
    {"de": "AV-02", "a": "AV-12", "km": 5.2, "min": 7, "limite_kmh": 60, "dirigida": false},
    {"de": "AV-12", "a": "PN-04", "km": 6.0, "min": 8, "limite_kmh": 60, "dirigida": false}
  ]
}

In [ ]:
%%writefile data/grafo_camaras_real.json
{
 "version": "real",
 "nota": "Red de fiscalización con ubicaciones de OpenStreetMap y tramos por calles reales (OSRM). Generada por calcular_tramos; no editar a mano.",
 "nodos": {
  "AV-01": {
   "nombre": "Av. Javier Prado",
   "lat": -12.09307,
   "lon": -77.03853,
   "limite_kmh": 60,
   "fuente": "OpenStreetMap: Av. Javier Prado Oeste, San Isidro"
  },
  "AV-08": {
   "nombre": "Av. Arequipa",
   "lat": -12.08891,
   "lon": -77.03438,
   "limite_kmh": 60,
   "fuente": "OpenStreetMap: Av. Arequipa 2450, Lince"
  },
  "AV-15": {
   "nombre": "Av. Angamos",
   "lat": -12.11385,
   "lon": -77.03187,
   "limite_kmh": 60,
   "fuente": "OpenStreetMap: Av. Angamos Oeste, Miraflores"
  },
  "AV-02": {
   "nombre": "Av. Salaverry",
   "lat": -12.07657,
   "lon": -77.04208,
   "limite_kmh": 60,
   "fuente": "OpenStreetMap: Av. General Salaverry, Jesús María"
  },
  "AV-12": {
   "nombre": "Av. Brasil",
   "lat": -12.06661,
   "lon": -77.04669,
   "limite_kmh": 60,
   "fuente": "OpenStreetMap: Av. Brasil, Jesús María"
  },
  "PN-04": {
   "nombre": "Panamericana Norte",
   "lat": -11.99714,
   "lon": -77.05989,
   "limite_kmh": 80,
   "fuente": "OpenStreetMap: Panamericana Norte, Independencia"
  }
 },
 "aristas": [
  {
   "de": "AV-01",
   "a": "AV-02",
   "km": 2.39,
   "min": 5.3,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.07657,
     -77.04209
    ],
    [
     -12.07663,
     -77.04149
    ],
    [
     -12.07641,
     -77.04153
    ],
    [
     -12.07652,
     -77.04232
    ],
    [
     -12.07969,
     -77.0424
    ],
    [
     -12.08003,
     -77.03958
    ],
    [
     -12.08972,
     -77.03821
    ],
    [
     -12.08984,
     -77.0391
    ],
    [
     -12.09307,
     -77.03853
    ]
   ]
  },
  {
   "de": "AV-01",
   "a": "AV-08",
   "km": 0.95,
   "min": 2.0,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.08888,
     -77.03417
    ],
    [
     -12.08914,
     -77.03413
    ],
    [
     -12.08984,
     -77.0391
    ],
    [
     -12.09308,
     -77.03863
    ],
    [
     -12.09307,
     -77.03853
    ]
   ]
  },
  {
   "de": "AV-01",
   "a": "AV-15",
   "km": 3.25,
   "min": 6.4,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.11385,
     -77.03187
    ],
    [
     -12.11376,
     -77.03135
    ],
    [
     -12.11381,
     -77.03239
    ],
    [
     -12.11166,
     -77.03238
    ],
    [
     -12.11162,
     -77.03015
    ],
    [
     -12.09705,
     -77.03256
    ],
    [
     -12.09712,
     -77.03361
    ],
    [
     -12.09641,
     -77.03553
    ],
    [
     -12.09579,
     -77.03824
    ],
    [
     -12.09307,
     -77.03853
    ]
   ]
  },
  {
   "de": "AV-02",
   "a": "AV-08",
   "km": 2.03,
   "min": 4.5,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.07657,
     -77.04209
    ],
    [
     -12.07698,
     -77.03804
    ],
    [
     -12.07683,
     -77.03677
    ],
    [
     -12.08874,
     -77.03501
    ],
    [
     -12.08863,
     -77.0342
    ],
    [
     -12.08888,
     -77.03417
    ]
   ]
  },
  {
   "de": "AV-02",
   "a": "AV-12",
   "km": 1.63,
   "min": 3.5,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.07657,
     -77.04209
    ],
    [
     -12.07545,
     -77.04203
    ],
    [
     -12.07392,
     -77.04182
    ],
    [
     -12.07341,
     -77.04168
    ],
    [
     -12.07216,
     -77.04111
    ],
    [
     -12.07148,
     -77.04403
    ],
    [
     -12.06825,
     -77.04806
    ],
    [
     -12.06661,
     -77.04669
    ]
   ]
  },
  {
   "de": "AV-02",
   "a": "PN-04",
   "km": 10.63,
   "min": 17.0,
   "limite_kmh": 80,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -11.99741,
     -77.05985
    ],
    [
     -11.99722,
     -77.05858
    ],
    [
     -12.00379,
     -77.05802
    ],
    [
     -12.00332,
     -77.05479
    ],
    [
     -12.02266,
     -77.05063
    ],
    [
     -12.02852,
     -77.04447
    ],
    [
     -12.06029,
     -77.0421
    ],
    [
     -12.06503,
     -77.03911
    ],
    [
     -12.06868,
     -77.03976
    ],
    [
     -12.07366,
     -77.04201
    ],
    [
     -12.07969,
     -77.0424
    ],
    [
     -12.07657,
     -77.04209
    ]
   ]
  },
  {
   "de": "AV-08",
   "a": "AV-12",
   "km": 3.87,
   "min": 7.3,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.06661,
     -77.04669
    ],
    [
     -12.06535,
     -77.04564
    ],
    [
     -12.06458,
     -77.03988
    ],
    [
     -12.06491,
     -77.03924
    ],
    [
     -12.0647,
     -77.03773
    ],
    [
     -12.06805,
     -77.03811
    ],
    [
     -12.08874,
     -77.03501
    ],
    [
     -12.08863,
     -77.0342
    ],
    [
     -12.08888,
     -77.03417
    ]
   ]
  },
  {
   "de": "AV-08",
   "a": "AV-15",
   "km": 3.32,
   "min": 6.3,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.08888,
     -77.03417
    ],
    [
     -12.08914,
     -77.03413
    ],
    [
     -12.08926,
     -77.03493
    ],
    [
     -12.0908,
     -77.03471
    ],
    [
     -12.09064,
     -77.03365
    ],
    [
     -12.09099,
     -77.03358
    ],
    [
     -12.09846,
     -77.03252
    ],
    [
     -12.1137,
     -77.02994
    ],
    [
     -12.11381,
     -77.03239
    ],
    [
     -12.11385,
     -77.03187
    ]
   ]
  },
  {
   "de": "AV-08",
   "a": "PN-04",
   "km": 11.53,
   "min": 18.6,
   "limite_kmh": 80,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -11.99741,
     -77.05985
    ],
    [
     -11.99722,
     -77.05858
    ],
    [
     -12.00379,
     -77.05802
    ],
    [
     -12.00332,
     -77.05479
    ],
    [
     -12.02266,
     -77.05063
    ],
    [
     -12.02846,
     -77.04448
    ],
    [
     -12.06029,
     -77.0421
    ],
    [
     -12.06483,
     -77.03939
    ],
    [
     -12.06474,
     -77.03772
    ],
    [
     -12.06764,
     -77.03817
    ],
    [
     -12.08874,
     -77.03501
    ],
    [
     -12.08888,
     -77.03417
    ]
   ]
  },
  {
   "de": "AV-12",
   "a": "AV-15",
   "km": 8.07,
   "min": 10.3,
   "limite_kmh": 60,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -12.06661,
     -77.04669
    ],
    [
     -12.06535,
     -77.04564
    ],
    [
     -12.06458,
     -77.03988
    ],
    [
     -12.06491,
     -77.03918
    ],
    [
     -12.06383,
     -77.0341
    ],
    [
     -12.08808,
     -77.02443
    ],
    [
     -12.09038,
     -77.02306
    ],
    [
     -12.09197,
     -77.02301
    ],
    [
     -12.10039,
     -77.02685
    ],
    [
     -12.1029,
     -77.02749
    ],
    [
     -12.1056,
     -77.02748
    ],
    [
     -12.1095,
     -77.02643
    ],
    [
     -12.11345,
     -77.02633
    ],
    [
     -12.11381,
     -77.03239
    ],
    [
     -12.11385,
     -77.03187
    ]
   ]
  },
  {
   "de": "AV-12",
   "a": "PN-04",
   "km": 9.45,
   "min": 15.3,
   "limite_kmh": 80,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -11.99741,
     -77.05985
    ],
    [
     -11.99722,
     -77.05858
    ],
    [
     -12.00379,
     -77.05802
    ],
    [
     -12.00332,
     -77.05479
    ],
    [
     -12.02266,
     -77.05063
    ],
    [
     -12.02852,
     -77.04447
    ],
    [
     -12.05955,
     -77.04176
    ],
    [
     -12.06001,
     -77.04212
    ],
    [
     -12.05931,
     -77.04444
    ],
    [
     -12.0654,
     -77.04956
    ],
    [
     -12.06727,
     -77.04725
    ],
    [
     -12.06661,
     -77.04669
    ]
   ]
  },
  {
   "de": "AV-15",
   "a": "PN-04",
   "km": 15.72,
   "min": 21.7,
   "limite_kmh": 80,
   "dirigida": false,
   "fuente": "OSRM (ruta directa)",
   "geometria": [
    [
     -11.99741,
     -77.05985
    ],
    [
     -11.99722,
     -77.05858
    ],
    [
     -12.00379,
     -77.05802
    ],
    [
     -12.00332,
     -77.05479
    ],
    [
     -12.02266,
     -77.05063
    ],
    [
     -12.02852,
     -77.04447
    ],
    [
     -12.06044,
     -77.04206
    ],
    [
     -12.0649,
     -77.0393
    ],
    [
     -12.06383,
     -77.0341
    ],
    [
     -12.09059,
     -77.023
    ],
    [
     -12.1029,
     -77.02749
    ],
    [
     -12.11345,
     -77.02633
    ],
    [
     -12.11385,
     -77.03187
    ]
   ]
  },
  {
   "de": "AV-01",
   "a": "PN-04",
   "km": 12.12,
   "min": 20.0,
   "limite_kmh": 80,
   "dirigida": false,
   "fuente": "OSRM (agregado para no superar la distancia real)",
   "geometria": [
    [
     -11.99741,
     -77.05985
    ],
    [
     -11.99722,
     -77.05858
    ],
    [
     -12.00379,
     -77.05802
    ],
    [
     -12.00332,
     -77.05479
    ],
    [
     -12.02266,
     -77.05063
    ],
    [
     -12.02852,
     -77.04447
    ],
    [
     -12.06029,
     -77.0421
    ],
    [
     -12.06503,
     -77.03911
    ],
    [
     -12.06868,
     -77.03976
    ],
    [
     -12.07366,
     -77.04201
    ],
    [
     -12.07969,
     -77.0424
    ],
    [
     -12.08003,
     -77.03958
    ],
    [
     -12.08972,
     -77.03821
    ],
    [
     -12.08984,
     -77.0391
    ],
    [
     -12.09307,
     -77.03853
    ]
   ]
  }
 ]
}

In [ ]:
%%writefile data/red_transito.json
{
 "version": "2026-10-16",
 "nota": "Cámaras públicas de tránsito de Lima (señal de la Municipalidad de Lima transmitida por Claro, claro.com.pe/traficoenvivo). Coordenadas de OpenStreetMap (aproximadas). Distancia y tiempo por calle de OSRM a flujo libre. Uso académico: conteo de vehículos, sin leer ni guardar placas.",
 "nodos": {
  "gallinazos": {
   "nombre": "Paradero Gallinazos",
   "distrito": "Puente Piedra",
   "ubicacion": "Km 26+800 Panamericana Norte",
   "lat": -11.8655,
   "lon": -77.0745,
   "fuente_coordenadas": "aproximada (manual)",
   "url": "https://live.smartechlatam.online/claro/gallinazos/index.m3u8"
  },
  "escuela_pnp": {
   "nombre": "Paradero Escuela PNP",
   "distrito": "Puente Piedra",
   "ubicacion": "Km 26+300 Panamericana Norte",
   "lat": -11.8695,
   "lon": -77.0735,
   "fuente_coordenadas": "aproximada (manual)",
   "url": "https://live.smartechlatam.online/claro/escuela_pnp/index.m3u8"
  },
  "prolongaciontacna": {
   "nombre": "Av. Prolongación Tacna",
   "distrito": "Rímac",
   "ubicacion": "Cuadra 1",
   "lat": -12.03424,
   "lon": -77.02927,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/prolongaciontacna/index.m3u8"
  },
  "lamarina": {
   "nombre": "Av. La Marina",
   "distrito": "Pueblo Libre",
   "ubicacion": "Cuadra 7",
   "lat": -12.08228,
   "lon": -77.06919,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/lamarina/index.m3u8"
  },
  "avfaustinocarrion": {
   "nombre": "Av. Faustino Sánchez Carrión",
   "distrito": "Jesús María",
   "ubicacion": "Cruce con Av. Punta del Este",
   "lat": -12.09134,
   "lon": -77.05565,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/avfaustinocarrion/index.m3u8"
  },
  "paseodelarepublica": {
   "nombre": "Av. Paseo de la República",
   "distrito": "Lince",
   "ubicacion": "N.° 1786, cruce con Av. Canadá",
   "lat": -12.08242,
   "lon": -77.02687,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/paseodelarepublica/index.m3u8"
  },
  "ejercito": {
   "nombre": "Av. del Ejército",
   "distrito": "Miraflores",
   "ubicacion": "Cuadra 12",
   "lat": -12.1164,
   "lon": -77.04527,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/ejercito/index.m3u8"
  },
  "angamos": {
   "nombre": "Av. Angamos",
   "distrito": "Surquillo",
   "ubicacion": "Cuadra 20, cruce con Pasaje Capitán Romano",
   "lat": -12.112,
   "lon": -77.00204,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/angamos/index.m3u8"
  },
  "republicapanama": {
   "nombre": "Av. República de Panamá",
   "distrito": "Surquillo",
   "ubicacion": "Cuadra 39, cruce con Pasaje Vicuña",
   "lat": -12.10568,
   "lon": -77.01858,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/republicapanama/index.m3u8"
  },
  "javierprado": {
   "nombre": "Av. Javier Prado Este",
   "distrito": "Santiago de Surco",
   "ubicacion": "Cuadra 45 (grifo Primax)",
   "lat": -12.08251,
   "lon": -76.9687,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/javierprado/index.m3u8"
  },
  "derby": {
   "nombre": "Av. El Derby",
   "distrito": "Santiago de Surco",
   "ubicacion": "Puerta 3 del Jockey Club",
   "lat": -12.09886,
   "lon": -76.97041,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/derby/index.m3u8"
  },
  "panamericana": {
   "nombre": "Panamericana Sur (Puente Atocongo)",
   "distrito": "Santiago de Surco",
   "ubicacion": "Puente Atocongo",
   "lat": -12.14963,
   "lon": -76.98313,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/panamericana/index.m3u8"
  },
  "plaza_villa": {
   "nombre": "Peaje Plaza Villa",
   "distrito": "Villa El Salvador",
   "ubicacion": "Km 30, Panamericana Sur",
   "lat": -12.22679,
   "lon": -76.97045,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/plaza_villa/index.m3u8"
  },
  "domo_quebrada_seca": {
   "nombre": "Domo Quebrada Seca",
   "distrito": "Punta Hermosa",
   "ubicacion": "Km 40, Panamericana Sur",
   "lat": -12.323,
   "lon": -76.837,
   "fuente_coordenadas": "aproximada (manual)",
   "url": "https://live.smartechlatam.online/claro/domo_quebrada_seca/index.m3u8"
  },
  "domo_punta_negra": {
   "nombre": "Peaje Punta Negra",
   "distrito": "Punta Negra",
   "ubicacion": "Km 48, Panamericana Sur",
   "lat": -12.36642,
   "lon": -76.78933,
   "fuente_coordenadas": "OpenStreetMap (ubicación de la vía, aproximada)",
   "url": "https://live.smartechlatam.online/claro/domo_punta_negra/index.m3u8"
  }
 },
 "aristas": [
  {
   "de": "gallinazos",
   "a": "escuela_pnp",
   "km": 0.62,
   "min": 1.1,
   "corredor": "Panamericana Norte",
   "geometria": [
    [
     -11.86532,
     -77.07466
    ],
    [
     -11.86513,
     -77.07444
    ],
    [
     -11.86571,
     -77.07398
    ],
    [
     -11.86754,
     -77.07277
    ],
    [
     -11.86799,
     -77.07346
    ],
    [
     -11.86815,
     -77.07356
    ],
    [
     -11.86879,
     -77.07378
    ],
    [
     -11.86887,
     -77.07385
    ],
    [
     -11.86934,
     -77.07335
    ]
   ]
  },
  {
   "de": "escuela_pnp",
   "a": "prolongaciontacna",
   "km": 21.64,
   "min": 25.3,
   "corredor": "Panamericana Norte / Vía de Evitamiento",
   "geometria": [
    [
     -11.86934,
     -77.07335
    ],
    [
     -11.87321,
     -77.06934
    ],
    [
     -11.90205,
     -77.06769
    ],
    [
     -11.91558,
     -77.0739
    ],
    [
     -11.94642,
     -77.07095
    ],
    [
     -12.00866,
     -77.06165
    ],
    [
     -12.00746,
     -77.05394
    ],
    [
     -12.02329,
     -77.05019
    ],
    [
     -12.02985,
     -77.04296
    ],
    [
     -12.02685,
     -77.04001
    ],
    [
     -12.03424,
     -77.02927
    ]
   ]
  },
  {
   "de": "prolongaciontacna",
   "a": "paseodelarepublica",
   "km": 6.59,
   "min": 10.6,
   "corredor": "Vía Expresa Paseo de la República",
   "geometria": [
    [
     -12.03424,
     -77.02927
    ],
    [
     -12.03347,
     -77.02898
    ],
    [
     -12.04856,
     -77.03914
    ],
    [
     -12.05417,
     -77.03848
    ],
    [
     -12.05437,
     -77.04062
    ],
    [
     -12.05609,
     -77.04047
    ],
    [
     -12.05587,
     -77.03648
    ],
    [
     -12.06009,
     -77.03596
    ],
    [
     -12.06168,
     -77.03478
    ],
    [
     -12.08252,
     -77.02686
    ],
    [
     -12.08212,
     -77.02551
    ],
    [
     -12.08242,
     -77.02687
    ]
   ]
  },
  {
   "de": "prolongaciontacna",
   "a": "avfaustinocarrion",
   "km": 7.93,
   "min": 13.6,
   "corredor": "Av. Brasil / Av. Salaverry",
   "geometria": [
    [
     -12.03424,
     -77.02927
    ],
    [
     -12.03347,
     -77.02898
    ],
    [
     -12.0486,
     -77.03916
    ],
    [
     -12.06426,
     -77.03764
    ],
    [
     -12.06461,
     -77.03945
    ],
    [
     -12.06848,
     -77.03969
    ],
    [
     -12.07382,
     -77.04204
    ],
    [
     -12.08044,
     -77.04259
    ],
    [
     -12.0856,
     -77.04938
    ],
    [
     -12.09289,
     -77.0533
    ],
    [
     -12.09133,
     -77.05565
    ]
   ]
  },
  {
   "de": "lamarina",
   "a": "avfaustinocarrion",
   "km": 1.83,
   "min": 2.7,
   "corredor": "Av. Sucre / Av. Faustino Sánchez Carrión",
   "geometria": [
    [
     -12.08228,
     -77.06919
    ],
    [
     -12.08188,
     -77.07029
    ],
    [
     -12.08203,
     -77.07035
    ],
    [
     -12.08368,
     -77.06579
    ],
    [
     -12.08401,
     -77.06512
    ],
    [
     -12.08435,
     -77.06475
    ],
    [
     -12.08555,
     -77.06432
    ],
    [
     -12.0867,
     -77.06295
    ],
    [
     -12.09316,
     -77.0532
    ],
    [
     -12.09301,
     -77.05312
    ],
    [
     -12.09133,
     -77.05565
    ]
   ]
  },
  {
   "de": "lamarina",
   "a": "ejercito",
   "km": 5.43,
   "min": 8.5,
   "corredor": "Av. La Marina / Av. del Ejército",
   "geometria": [
    [
     -12.08228,
     -77.06919
    ],
    [
     -12.08203,
     -77.07035
    ],
    [
     -12.08401,
     -77.06512
    ],
    [
     -12.08555,
     -77.06432
    ],
    [
     -12.0867,
     -77.06295
    ],
    [
     -12.09482,
     -77.05071
    ],
    [
     -12.09888,
     -77.05011
    ],
    [
     -12.09897,
     -77.05115
    ],
    [
     -12.10499,
     -77.05193
    ],
    [
     -12.10623,
     -77.05247
    ],
    [
     -12.10776,
     -77.05393
    ],
    [
     -12.11679,
     -77.045
    ],
    [
     -12.1164,
     -77.04527
    ]
   ]
  },
  {
   "de": "avfaustinocarrion",
   "a": "paseodelarepublica",
   "km": 3.8,
   "min": 7.5,
   "corredor": "Av. Javier Prado Oeste / Av. Canadá",
   "geometria": [
    [
     -12.09133,
     -77.05565
    ],
    [
     -12.09113,
     -77.05595
    ],
    [
     -12.09102,
     -77.05583
    ],
    [
     -12.0882,
     -77.05103
    ],
    [
     -12.08812,
     -77.05063
    ],
    [
     -12.08629,
     -77.04963
    ],
    [
     -12.08545,
     -77.0489
    ],
    [
     -12.08251,
     -77.02681
    ],
    [
     -12.08231,
     -77.0256
    ],
    [
     -12.08212,
     -77.02551
    ],
    [
     -12.08242,
     -77.02687
    ]
   ]
  },
  {
   "de": "avfaustinocarrion",
   "a": "ejercito",
   "km": 3.6,
   "min": 5.9,
   "corredor": "Av. Salaverry / Av. Pershing",
   "geometria": [
    [
     -12.09133,
     -77.05565
    ],
    [
     -12.09102,
     -77.05583
    ],
    [
     -12.08813,
     -77.05087
    ],
    [
     -12.1029,
     -77.05844
    ],
    [
     -12.11679,
     -77.045
    ],
    [
     -12.1164,
     -77.04527
    ]
   ]
  },
  {
   "de": "paseodelarepublica",
   "a": "republicapanama",
   "km": 2.86,
   "min": 4.6,
   "corredor": "Vía Expresa / Av. República de Panamá",
   "geometria": [
    [
     -12.08242,
     -77.02687
    ],
    [
     -12.08686,
     -77.0251
    ],
    [
     -12.09002,
     -77.02319
    ],
    [
     -12.09102,
     -77.02292
    ],
    [
     -12.09216,
     -77.02308
    ],
    [
     -12.0968,
     -77.02538
    ],
    [
     -12.09746,
     -77.02059
    ],
    [
     -12.1019,
     -77.01889
    ],
    [
     -12.10556,
     -77.0187
    ],
    [
     -12.10568,
     -77.01858
    ]
   ]
  },
  {
   "de": "ejercito",
   "a": "republicapanama",
   "km": 4.11,
   "min": 7.2,
   "corredor": "Av. Angamos Oeste / Av. Ricardo Palma",
   "geometria": [
    [
     -12.1164,
     -77.04527
    ],
    [
     -12.11607,
     -77.04569
    ],
    [
     -12.11679,
     -77.045
    ],
    [
     -12.11303,
     -77.04105
    ],
    [
     -12.11414,
     -77.03992
    ],
    [
     -12.11374,
     -77.02872
    ],
    [
     -12.11303,
     -77.01842
    ],
    [
     -12.10568,
     -77.01858
    ]
   ]
  },
  {
   "de": "republicapanama",
   "a": "angamos",
   "km": 2.3,
   "min": 3.3,
   "corredor": "Av. Angamos",
   "geometria": [
    [
     -12.10568,
     -77.01858
    ],
    [
     -12.10501,
     -77.01859
    ],
    [
     -12.10492,
     -77.01838
    ],
    [
     -12.10831,
     -77.01532
    ],
    [
     -12.11262,
     -77.0121
    ],
    [
     -12.11203,
     -77.00366
    ],
    [
     -12.11202,
     -77.00342
    ],
    [
     -12.1121,
     -77.0034
    ],
    [
     -12.112,
     -77.00204
    ]
   ]
  },
  {
   "de": "angamos",
   "a": "javierprado",
   "km": 6.75,
   "min": 8.8,
   "corredor": "Av. Panamericana Sur / Av. Javier Prado Este",
   "geometria": [
    [
     -12.112,
     -77.00204
    ],
    [
     -12.11187,
     -77.00003
    ],
    [
     -12.08828,
     -77.00346
    ],
    [
     -12.086,
     -76.98791
    ],
    [
     -12.08516,
     -76.98511
    ],
    [
     -12.08375,
     -76.96991
    ],
    [
     -12.08251,
     -76.9687
    ]
   ]
  },
  {
   "de": "paseodelarepublica",
   "a": "javierprado",
   "km": 7.41,
   "min": 9.7,
   "corredor": "Av. Javier Prado Este",
   "geometria": [
    [
     -12.08242,
     -77.02687
    ],
    [
     -12.08686,
     -77.0251
    ],
    [
     -12.09016,
     -77.02313
    ],
    [
     -12.09139,
     -77.02292
    ],
    [
     -12.09202,
     -77.02331
    ],
    [
     -12.09127,
     -77.02365
    ],
    [
     -12.09088,
     -77.02304
    ],
    [
     -12.08718,
     -76.99553
    ],
    [
     -12.086,
     -76.98791
    ],
    [
     -12.08516,
     -76.98511
    ],
    [
     -12.08375,
     -76.96991
    ],
    [
     -12.08251,
     -76.9687
    ]
   ]
  },
  {
   "de": "javierprado",
   "a": "derby",
   "km": 2.45,
   "min": 3.4,
   "corredor": "Av. Javier Prado Este / Av. El Derby",
   "geometria": [
    [
     -12.08251,
     -76.9687
    ],
    [
     -12.08183,
     -76.96764
    ],
    [
     -12.0814,
     -76.96789
    ],
    [
     -12.08195,
     -76.96812
    ],
    [
     -12.08285,
     -76.96954
    ],
    [
     -12.08295,
     -76.97026
    ],
    [
     -12.08343,
     -76.97045
    ],
    [
     -12.08403,
     -76.96991
    ],
    [
     -12.08513,
     -76.96936
    ],
    [
     -12.08634,
     -76.96775
    ],
    [
     -12.08717,
     -76.96722
    ],
    [
     -12.08799,
     -76.96706
    ],
    [
     -12.09153,
     -76.96715
    ],
    [
     -12.09328,
     -76.96747
    ],
    [
     -12.09499,
     -76.96815
    ],
    [
     -12.09686,
     -76.96991
    ],
    [
     -12.099,
     -76.9692
    ],
    [
     -12.09886,
     -76.97041
    ]
   ]
  },
  {
   "de": "angamos",
   "a": "panamericana",
   "km": 6.72,
   "min": 9.7,
   "corredor": "Panamericana Sur",
   "geometria": [
    [
     -12.112,
     -77.00204
    ],
    [
     -12.11189,
     -77.0003
    ],
    [
     -12.11848,
     -76.99859
    ],
    [
     -12.12071,
     -76.99837
    ],
    [
     -12.12224,
     -76.9986
    ],
    [
     -12.12904,
     -77.00154
    ],
    [
     -12.1296,
     -77.00074
    ],
    [
     -12.13998,
     -76.99335
    ],
    [
     -12.14699,
     -76.98743
    ],
    [
     -12.14956,
     -76.98347
    ],
    [
     -12.15053,
     -76.9812
    ],
    [
     -12.15377,
     -76.98343
    ],
    [
     -12.15501,
     -76.98342
    ],
    [
     -12.15328,
     -76.98376
    ],
    [
     -12.14965,
     -76.98307
    ]
   ]
  },
  {
   "de": "derby",
   "a": "panamericana",
   "km": 6.22,
   "min": 7.4,
   "corredor": "Av. Primavera / Panamericana Sur",
   "geometria": [
    [
     -12.09886,
     -76.97041
    ],
    [
     -12.0991,
     -76.96923
    ],
    [
     -12.10976,
     -76.97282
    ],
    [
     -12.11017,
     -76.97876
    ],
    [
     -12.10969,
     -76.97913
    ],
    [
     -12.10931,
     -76.97874
    ],
    [
     -12.10978,
     -76.97839
    ],
    [
     -12.11723,
     -76.97738
    ],
    [
     -12.12732,
     -76.97705
    ],
    [
     -12.13683,
     -76.98043
    ],
    [
     -12.14642,
     -76.98221
    ],
    [
     -12.15026,
     -76.98362
    ],
    [
     -12.15042,
     -76.98429
    ],
    [
     -12.14981,
     -76.98463
    ],
    [
     -12.14938,
     -76.98386
    ],
    [
     -12.1506,
     -76.98116
    ],
    [
     -12.15377,
     -76.98343
    ],
    [
     -12.15501,
     -76.98342
    ],
    [
     -12.15328,
     -76.98376
    ],
    [
     -12.14965,
     -76.98307
    ]
   ]
  },
  {
   "de": "panamericana",
   "a": "plaza_villa",
   "km": 10.48,
   "min": 11.1,
   "corredor": "Panamericana Sur",
   "geometria": [
    [
     -12.14965,
     -76.98307
    ],
    [
     -12.14613,
     -76.98174
    ],
    [
     -12.14836,
     -76.98237
    ],
    [
     -12.14986,
     -76.98138
    ],
    [
     -12.15002,
     -76.98211
    ],
    [
     -12.1488,
     -76.98393
    ],
    [
     -12.14901,
     -76.98315
    ],
    [
     -12.15531,
     -76.98378
    ],
    [
     -12.19206,
     -76.97272
    ],
    [
     -12.21901,
     -76.97703
    ],
    [
     -12.22119,
     -76.97597
    ],
    [
     -12.22679,
     -76.97045
    ]
   ]
  },
  {
   "de": "plaza_villa",
   "a": "domo_quebrada_seca",
   "km": 21.17,
   "min": 19.3,
   "corredor": "Panamericana Sur",
   "geometria": [
    [
     -12.22679,
     -76.97045
    ],
    [
     -12.23106,
     -76.96599
    ],
    [
     -12.24737,
     -76.93472
    ],
    [
     -12.25319,
     -76.92688
    ],
    [
     -12.2951,
     -76.8597
    ],
    [
     -12.30425,
     -76.84073
    ],
    [
     -12.30887,
     -76.83639
    ],
    [
     -12.31581,
     -76.83176
    ],
    [
     -12.31668,
     -76.83222
    ],
    [
     -12.32479,
     -76.82679
    ],
    [
     -12.32761,
     -76.83107
    ],
    [
     -12.32839,
     -76.83121
    ],
    [
     -12.32845,
     -76.83326
    ],
    [
     -12.32815,
     -76.83394
    ],
    [
     -12.32714,
     -76.83365
    ],
    [
     -12.32464,
     -76.83447
    ],
    [
     -12.3239,
     -76.837
    ],
    [
     -12.323,
     -76.83702
    ]
   ]
  },
  {
   "de": "domo_quebrada_seca",
   "a": "domo_punta_negra",
   "km": 8.1,
   "min": 9.9,
   "corredor": "Panamericana Sur",
   "geometria": [
    [
     -12.323,
     -76.83702
    ],
    [
     -12.3239,
     -76.837
    ],
    [
     -12.32464,
     -76.83447
    ],
    [
     -12.32714,
     -76.83365
    ],
    [
     -12.32815,
     -76.83394
    ],
    [
     -12.32844,
     -76.83331
    ],
    [
     -12.32839,
     -76.83121
    ],
    [
     -12.3277,
     -76.83112
    ],
    [
     -12.32479,
     -76.82679
    ],
    [
     -12.32843,
     -76.82428
    ],
    [
     -12.31691,
     -76.83196
    ],
    [
     -12.31682,
     -76.83109
    ],
    [
     -12.34443,
     -76.8126
    ],
    [
     -12.37445,
     -76.78182
    ],
    [
     -12.37378,
     -76.78162
    ],
    [
     -12.3664,
     -76.78931
    ]
   ]
  }
 ]
}

## 2. Importar y funciones de apoyo

In [ ]:
import json, random, time, timeit
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import networkx as nx

import grafo
from grafo import (a_estrella, bfs, cargar, construir, costo_ruta, dfs, misma_placa,
                   nula, recta_km, recta_min)

print("Paquete grafo:", ", ".join(grafo.__all__))
print("NetworkX", nx.__version__)


def a_networkx(g):
    """Convierte nuestra lista de adyacencia en un grafo de NetworkX (referencia independiente)."""
    G = nx.DiGraph()
    for a, vecinos in g.items():
        G.add_node(a)
        for b, w in vecinos:
            G.add_edge(a, b, **{k: v for k, v in w.items() if v is not None})
    return G


def posiciones(nodos):
    if all("x_km" in n for n in nodos.values()):
        return {c: (n["x_km"], n["y_km"]) for c, n in nodos.items()}
    return {c: (n["lon"], n["lat"]) for c, n in nodos.items()}


def dibujar(g, nodos, ruta=None, colores=None, titulo="", peso="km", etiquetas_aristas=True,
            tam=(8, 5), tam_nodo=900, enfoque=None):
    """enfoque: lista de cámaras a encuadrar (acerca la vista a esa zona)."""
    G = a_networkx(g).to_undirected()
    pos = posiciones(nodos)
    plt.figure(figsize=tam)
    nx.draw_networkx_edges(G, pos, edge_color="#bbbbbb", width=1.5)
    if ruta:
        nx.draw_networkx_edges(G, pos, edgelist=list(zip(ruta, ruta[1:])), edge_color="#d62728", width=4)
    color = [colores.get(n, "#dddddd") for n in G.nodes] if colores else "#9ecae1"
    nx.draw_networkx_nodes(G, pos, node_color=color, node_size=tam_nodo, edgecolors="#333333")
    nx.draw_networkx_labels(G, pos, font_size=7)
    if etiquetas_aristas:
        nx.draw_networkx_edge_labels(G, pos, font_size=7,
                                     edge_labels={(a, b): round(d[peso], 1) for a, b, d in G.edges(data=True)})
    if enfoque:
        xs, ys = zip(*(pos[c] for c in enfoque))
        mx, my = (max(xs) - min(xs)) * 0.12, (max(ys) - min(ys)) * 0.12
        plt.xlim(min(xs) - mx, max(xs) + mx)
        plt.ylim(min(ys) - my, max(ys) + my)
    plt.title(titulo)
    plt.axis("off")
    plt.show()

## 3. Cargar y validar la red esquemática (Tabla I)
`cargar` valida que la distancia en línea recta entre cámaras conectadas nunca supere la longitud del tramo: es la condición para que la heurística de A* sea **admisible**.

In [ ]:
g, nodos = cargar("data/grafo_camaras.json")
for cam, vecinos in g.items():
    print(f"{cam} ({nodos[cam]['nombre']}):", [(v, w["km"]) for v, w in vecinos])
dibujar(g, nodos, titulo="Red de cámaras de evaluación (km por tramo)")

## 4. BFS — cámaras a ≤ k saltos (P1)
Cuando una cámara lee una placa, BFS indica en qué cámaras vecinas buscar la misma placa.

In [ ]:
saltos = bfs(g, "AV-01", k=2)
print("BFS desde AV-01, k = 2:", saltos)

ref = nx.single_source_shortest_path_length(a_networkx(g), "AV-01", cutoff=2)
print("Igual a NetworkX:", saltos == ref)

paleta = {0: "#d62728", 1: "#ff9f40", 2: "#ffe08a"}
dibujar(g, nodos, colores={c: paleta[s] for c, s in saltos.items()},
        titulo="BFS desde AV-01 (rojo = origen, naranja = 1 salto, amarillo = 2 saltos)")

## 5. DFS acotado — trayectorias posibles (P2)

In [ ]:
rutas = dfs(g, "AV-01", "PN-04", max_prof=6)
for r in sorted(rutas, key=lambda r: costo_ruta(g, r)):
    print(" → ".join(r), f"= {costo_ruta(g, r):.1f} km")

ref = nx.all_simple_paths(a_networkx(g), "AV-01", "PN-04", cutoff=6)
print("Igual a NetworkX:", sorted(map(tuple, rutas)) == sorted(map(tuple, ref)))

## 6. A* — ruta mínima con y sin heurística (P3, P4)
Con `h = 0`, A* se comporta como Dijkstra. Con la heurística de línea recta explora menos nodos y obtiene el mismo costo.

In [ ]:
h = recta_km(nodos)
for nombre, heur in [("h = 0 (Dijkstra)", nula), ("h = línea recta", h)]:
    traza = []
    r = a_estrella(g, "AV-01", "AV-08", heur, traza=traza)
    print(f"{nombre:18} ruta {r.ruta}, {r.costo} km, nodos explorados {r.explorados}, orden {traza}")

# P3: los 30 pares de cámaras frente a NetworkX
G = a_networkx(g)
pares = [(a, b) for a in g for b in g if a != b]
correctos = sum(
    abs(a_estrella(g, a, b).costo - nx.dijkstra_path_length(G, a, b, weight="km")) < 1e-9 and
    abs(a_estrella(g, a, b, h).costo - nx.dijkstra_path_length(G, a, b, weight="km")) < 1e-9
    for a, b in pares)
print(f"\nCosto correcto en {correctos}/{len(pares)} pares, con h = 0 y con heurística")

r = a_estrella(g, "AV-01", "PN-04", h)
dibujar(g, nodos, ruta=r.ruta, titulo=f"A* de AV-01 a PN-04: {' → '.join(r.ruta)} ({r.costo:.1f} km)")

## 7. Coincidencia de placas entre cámaras (P5)
Las placas que difieren en caracteres que el OCR confunde (B/8, O/0…) o en un solo carácter quedan **a verificar**.

In [ ]:
for leida in ["ABC-123", "A8C-123", "ABC-128", "D4F-582"]:
    print(f"ABC-123 vs {leida}: {misma_placa('ABC-123', leida)}")

## 8. Tiempo por ejecución (P6)
Los tiempos dependen del procesador; en el paper se midieron en un AMD Ryzen 5 9600X.

In [ ]:
def us(f, n=20000):
    return timeit.timeit(f, number=n) / n * 1e6

print(f"BFS          {us(lambda: bfs(g, 'AV-01', 2)):.1f} µs")
print(f"DFS          {us(lambda: dfs(g, 'AV-01', 'PN-04')):.1f} µs")
print(f"A* (h = 0)   {us(lambda: a_estrella(g, 'AV-01', 'PN-04')):.1f} µs")
print(f"A* (recta)   {us(lambda: a_estrella(g, 'AV-01', 'PN-04', h)):.1f} µs")

## 9. Escalabilidad: 400 cámaras, 760 tramos, 300 consultas (P7)
Rejilla sintética de 20 × 20 cámaras; cada tramo mide entre 1.0 y 1.6 veces la distancia recta, para que la heurística siga siendo admisible. Semilla 42, como en el paper.

In [ ]:
def red_rejilla(n=20, paso_km=0.8, semilla=42):
    rnd = random.Random(semilla)
    nodos_r = {f"C{i}_{j}": {"x_km": i * paso_km, "y_km": j * paso_km} for i in range(n) for j in range(n)}
    aristas = []
    for i in range(n):
        for j in range(n):
            for di, dj in ((1, 0), (0, 1)):
                if i + di < n and j + dj < n:
                    km = round(paso_km * rnd.uniform(1.0, 1.6), 2)
                    aristas.append({"de": f"C{i}_{j}", "a": f"C{i+di}_{j+dj}", "km": km, "min": km * 1.5})
    return {"nodos": nodos_r, "aristas": aristas}, rnd


datos, rnd = red_rejilla()
gg, hh = construir(datos), recta_km(datos["nodos"])
Gr = nx.Graph()
for a in datos["aristas"]:
    Gr.add_edge(a["de"], a["a"], km=a["km"])

claves, consultas = list(datos["nodos"]), 300
exp0 = exph = t0 = th = correctos = 0
for _ in range(consultas):
    a, b = rnd.sample(claves, 2)
    s = time.perf_counter(); r0 = a_estrella(gg, a, b, nula); t0 += time.perf_counter() - s
    s = time.perf_counter(); rh = a_estrella(gg, a, b, hh); th += time.perf_counter() - s
    ref = nx.dijkstra_path_length(Gr, a, b, weight="km")
    correctos += abs(r0.costo - ref) < 1e-9 and abs(rh.costo - ref) < 1e-9
    exp0 += r0.explorados
    exph += rh.explorados

print(f"Cámaras {len(datos['nodos'])}, tramos {len(datos['aristas'])}, consultas {consultas}")
print(f"Costo correcto: {correctos}/{consultas}")
print(f"Nodos explorados: {exp0/consultas:.1f} (h = 0) → {exph/consultas:.1f} (recta), "
      f"reducción {100*(1-exph/exp0):.1f} %")
print(f"Tiempo por consulta: {t0/consultas*1000:.3f} ms → {th/consultas*1000:.3f} ms")

plt.figure(figsize=(5, 3))
plt.bar(["A* con h = 0", "A* con línea recta"], [exp0 / consultas, exph / consultas], color=["#bbbbbb", "#1f77b4"])
plt.ylabel("Nodos explorados (promedio)")
plt.title("400 cámaras, 300 consultas")
plt.show()

## 10. Red real (OpenStreetMap + OSRM)
Las mismas 6 cámaras con coordenadas reales (lat/lon): la heurística usa la distancia de Haversine. El tramo AV-01–AV-08 mide 0.95 km por calles, frente a 3.0 km en la red esquemática.

In [ ]:
gr, nr = cargar("data/grafo_camaras_real.json")
r = a_estrella(gr, "AV-01", "AV-08", recta_km(nr))
print(f"A* AV-01 → AV-08 por calles: {' → '.join(r.ruta)}, {r.costo:.2f} km")
dibujar(gr, nr, ruta=r.ruta, titulo="Red real: A* de AV-01 a AV-08 (km por calles)")

## 11. Cámaras públicas: ruta más rápida según el tráfico
En el sistema, el conteo con YOLOv8n + ByteTrack da un **factor de congestión** por cámara (vehículos visibles ÷ lo
normal en esa cámara, entre 1 y 3). El tiempo de cada tramo se multiplica por el promedio del factor de sus dos
cámaras, y A* busca la ruta más rápida con la heurística *línea recta ÷ velocidad máxima de la red*, que nunca
sobreestima. Aquí los factores son de **ejemplo**; cámbialos para ver cómo cambia la ruta.

In [ ]:
datos_t = json.load(open("data/red_transito.json", encoding="utf-8"))
print("Errores de validación:", grafo.validar(datos_t) or "ninguno")


def ruta_mas_rapida(datos, factores, origen, destino):
    aristas, vmax = [], 1.0
    for a in datos["aristas"]:
        f = (factores.get(a["de"], 1.0) + factores.get(a["a"], 1.0)) / 2
        aristas.append({**a, "min": a["min"] * f})
        vmax = max(vmax, a["km"] / (a["min"] / 60))   # velocidad máxima a flujo libre
    g_t = construir({"nodos": datos["nodos"], "aristas": aristas})
    h_t = recta_min(datos["nodos"], vmax_kmh=vmax)
    return a_estrella(g_t, origen, destino, h=h_t, peso="min"), g_t, h_t


libre, g_t, h_t = ruta_mas_rapida(datos_t, {}, "lamarina", "panamericana")
# Admisibilidad: la heurística no supera el tiempo de ningún arco
arcos = [(a, b, w["min"]) for a, vec in g_t.items() for b, w in vec]
print(f"Heurística admisible en {sum(h_t(a, b) <= m + 1e-9 for a, b, m in arcos)}/{len(arcos)} arcos")

congestion = {"lamarina": 1.4, "ejercito": 1.3, "angamos": 2.2, "republicapanama": 1.8}  # ejemplo
ahora, g_c, _ = ruta_mas_rapida(datos_t, congestion, "lamarina", "panamericana")
print(f"Flujo libre: {' → '.join(libre.ruta)} = {libre.costo:.1f} min")
print(f"Con tráfico: {' → '.join(ahora.ruta)} = {ahora.costo:.1f} min")

paleta = lambda f: "#d62728" if f >= 1.6 else "#ff9f40" if f >= 1.25 else "#7fd17f"
centro = ["lamarina", "avfaustinocarrion", "paseodelarepublica", "ejercito", "republicapanama",
          "angamos", "javierprado", "derby", "panamericana"]
dibujar(g_c, datos_t["nodos"], ruta=ahora.ruta, peso="min", tam=(10, 8), tam_nodo=350, enfoque=centro,
        colores={c: paleta(congestion.get(c, 1.0)) for c in datos_t["nodos"]},
        titulo="Cámaras públicas (zona central): congestión de ejemplo, minutos por tramo y ruta más rápida con A*")

## Anexo (trabajo futuro): velocidad promedio entre cámaras
Prototipo verificado **solo con capturas simuladas**; su validación con cámaras propias queda para la siguiente fase.
Usa la distancia mínima de A* y el mayor tiempo posible (Δt + 2ε, con ε = 2 s por desfase de reloj):
v = 3600·d_min / (Δt + 2ε), una cota inferior de la velocidad real, a favor del conductor.

In [ ]:
t = datetime(2026, 10, 15, 8, 0, 0)
for segundos in (120, 200):
    v = grafo.velocidad_por_tramo(g, "AV-01", t, "AV-08", t + timedelta(seconds=segundos), h=h)
    print(f"Δt = {segundos} s: medida {v.v_medida} km/h, conservadora {v.v_conservadora} km/h, "
          f"límite {v.limite}, ¿infracción? {v.infraccion}")